# Phase 2: GeoJSON streaming export

This notebook implements the Phase 2 processing pipeline. Here, we add synthetic latitude and longitude to the telemetry stream, which allows our simulated agents to have coordinate trajectories. The stream is synchronously processed via Spark Structured Streaming, exporting each micro-batch into a `GeoJSON` `FeatureCollection` format for geospatial analysis.

*Note*: The coordinates are simulated programmatically for pipeline demonstration purposes and do not represent actual GPS observations.


In [1]:
import os
import sys
import logging
from pathlib import Path
from pyspark.sql import SparkSession
import sys; sys.path.append(os.path.abspath('..'))
from src.step_08_bootstrapping import setup_winutils
# Climb up from the notebook's folder to find the true project workspace root
notebook_dir = Path(os.getcwd())
PROJECT_ROOT = notebook_dir.parent if (notebook_dir.parent / "src").exists() else notebook_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
# Configure Windows-specific local Spark settings dynamically
if os.name == 'nt':
    setup_winutils(PROJECT_ROOT)
    os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
# Check if an active Spark session already exists or is configured in the environment.
active_session = SparkSession.getActiveSession()
if active_session is not None:
    spark = active_session
    logging.info("Reusing active Spark Session.")
else:
    logging.info("Spawning adaptive geospatial Spark Session environment...")
    spark_builder = (
        SparkSession.builder
        .appName('Geospatial-Streaming-Export')
        .config('spark.sql.shuffle.partitions', '4')
    )
    if os.name == 'nt':
        spark_builder = (
            spark_builder
            .config('spark.driver.host', '127.0.0.1')
            .config('spark.pyspark.python', sys.executable)
            .config('spark.pyspark.driver.python', sys.executable)
        )
    master_url = os.environ.get("SPARK_MASTER")
    if not master_url and not any(env.startswith("SPARK_") for env in os.environ):
        spark_builder = spark_builder.master("local[*]") \
                                     .config("spark.driver.memory", "4g")
    spark = spark_builder.getOrCreate()
logging.info(f'Project root: {PROJECT_ROOT}')


2026-07-08 23:44:19,887 - INFO - Hadoop environment path configuration active: HADOOP_HOME=c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\winutils
2026-07-08 23:44:19,891 - INFO - Spawning adaptive geospatial Spark Session environment...


2026-07-08 23:44:30,290 - INFO - Project root: c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling


## 2.1 Spatial trajectory generation and coordinate projection

As our sensor data time-series has a dimension of 1, for a 2D geographical space the pipeline attaches the synthetically generated latitude ($\phi$) and longitude ($\lambda$) to each agent.

#### 1. Velocity Models
An agent's speed $v$ is determined by its classified activity state:
$$v = \begin{cases} 1.4\text{ m/s} & \text{if Activity} = \text{"walk"} \\ 4.5\text{ m/s} & \text{if Activity} = \text{"bike"} \\ 0.0\text{ m/s} & \text{otherwise} \end{cases}$$

#### 2. Angular Heading Direction
The walking or cycling heading direction $\theta$ (in radians) is determined by the agent's identifier modulo 16, dividing the compass into 16 discrete angles:
$$\theta = (\text{Agent\_ID} \pmod{16}) \times \frac{2\pi}{16}$$

#### 3. Spatial Displacement Mapping
Given the elapsed time $t$ since the base timestamp, the displacement in metres along the East ($\Delta x$) and North ($\Delta y$) axes is calculated as:
$$\Delta x = v \cdot t \cdot \cos(\theta)$$
$$\Delta y = v \cdot t \cdot \sin(\theta)$$

Using a local projection centred on Vienna ($\phi_{\text{start}} = 48.20849^\circ\text{ N}$, $\lambda_{\text{start}} = 16.37208^\circ\text{ E}$), we convert these displacements to degree coordinates:
$$\phi = \phi_{\text{start}} + \frac{\Delta y}{M_{\text{lat}}}$$
$$\lambda = \lambda_{\text{start}} + \frac{\Delta x}{M_{\text{lon}}}$$
where the metres-to-degrees conversion factors are defined by:
$$M_{\text{lat}} = 111,320.0 \text{ m/degree}$$
$$M_{\text{lon}} = 111,320.0 \cdot \cos(\phi_{\text{start}}) \text{ m/degree}$$


> [!NOTE]
> **Memory efficiency justification**
> The following simulation triggers `run_geospatial_streaming_simulation`, which downloads Vienna's spatial layers, snaps agent starting points, and resolves routes via `OSMnx`. It collects the distinct list of agents (limited to 150 synthetic subjects) to the driver to calculate spatial path coordinates. Collecting this small metadata list is computationally safe and does not cause driver memory overflow. The streaming coordinates are then iterated using `toLocalIterator()` to write the GeoJSON feature batches page-by-page, preventing out-of-memory errors.


In [2]:
from src.step_11_geospatial_streaming import run_geospatial_streaming_simulation
output_dir = run_geospatial_streaming_simulation(spark, PROJECT_ROOT)
geojson_files = sorted(output_dir.glob('*.geojson'))
print(f'Created {len(geojson_files)} GeoJSON micro-batch files in {output_dir}')
geojson_files[:3]


2026-07-08 23:45:05,997 - INFO - Layer already cached: vienna_districts.geojson
2026-07-08 23:45:06,005 - INFO - Layer already cached: vienna_pedestrian_zones.geojson
2026-07-08 23:45:06,009 - INFO - Layer already cached: vienna_bike_paths.geojson
2026-07-08 23:45:14,204 - INFO - Loading cached street graph: vienna_walk_network.graphml
2026-07-08 23:50:53,059 - INFO - Callback Server Starting
2026-07-08 23:50:53,110 - INFO - Socket listening on ('127.0.0.1', 56190)
2026-07-08 23:51:01,146 - INFO - Python Server ready to receive messages
2026-07-08 23:51:01,172 - INFO - Received command c on object id p0
2026-07-08 23:51:20,291 - INFO - GeoJSON micro-batch written to c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\geospatial_output\telemetry_batch_00000.geojson
2026-07-08 23:51:22,509 - INFO - Received command c on object id p0
2026-07-08 23:51:43,628 - INFO - GeoJSON micro-batch written to c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturatio

Created 10 GeoJSON micro-batch files in c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\geospatial_output


[WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_batch_00000.geojson'),
 WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_batch_00001.geojson'),
 WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_batch_00002.geojson')]

In [3]:
import json
with geojson_files[0].open('r', encoding='utf-8') as file_handle:
    first_batch = json.load(file_handle)
print(first_batch['type'])
print(f"Features in first batch: {len(first_batch['features'])}")
first_batch['features'][0]


FeatureCollection
Features in first batch: 150002


{'type': 'Feature',
 'geometry': {'type': 'Point', 'coordinates': [16.3742637, 48.1840581]},
 'properties': {'Agent_ID': 0,
  'Timestamp': 1700000000200,
  'Activity': 'walk',
  'ax': 1.7173888656001903,
  'ay': -0.5964573181081554,
  'az': 3.2621128736323373,
  'gx': 0.690795441661439,
  'gy': 0.3216134994160901,
  'gz': 0.07895205532422053,
  'coordinate_source': 'synthetic'}}

## 2.2 Interpretation and limitations

The generated GeoJSON files demonstrate how a Spark Structured Streaming pipeline can attach a geographical representation to synthetic telemetry. The coordinates are generated from explicit speed and direction assumptions; they are not observed GPS positions and are not snapped to official Vienna transport infrastructure. Consequently, the output is suitable for demonstrating data processing and visualisation, but not for drawing conclusions about actual Vienna mobility or congestion.


In [4]:
try:
    logging.info("Shutting down Spark Session...")
finally:
    spark.stop()
    logging.info("Spark Session terminated successfully.")


2026-07-08 23:55:03,972 - INFO - Shutting down Spark Session...
2026-07-08 23:55:04,308 - INFO - Spark Session terminated successfully.


## References

*   **Apache Spark. (n.d.).** *Spark Streaming*. Apache Software Foundation. Retrieved from https://spark.apache.org/streaming/
    *Annotation*: Compute framework powering our Structured Streaming micro-batch pipelines.
*   **Zaharia, M., Xin, R. S., Wendell, P., Das, T., Armbrust, M., Dave, A., Meng, X., Rosen, J., Venkataraman, S., Franklin, M. J., Ghodsi, A., Gonzalez, J., Shenker, S., & Stoica, I. (2016).** Apache Spark: A unified engine for big data processing. *Communications of the ACM*, *59*(11), 56-65. https://doi.org/10.1145/2934664
    *Annotation*: Distributed engine used for windowing aggregations and streaming.
*   **Boeing, G. (2017).** OSMnx: New methods for acquiring, constructing, analyzing, and visualizing complex street networks. *Computers, Environment and Urban Systems*, *65*, 126-139. https://doi.org/10.1016/j.compenvurbsys.2017.05.004
    *Annotation*: Used for Vienna network graph topological modeling and routing.
*   **OpenStreetMap contributors. (2026).** *Planet OSM* [Data set]. OpenStreetMap. https://www.openstreetmap.org/
    *Annotation*: Geographic road network layers under ODbL license.
